# 💉 Notebook 1: Knowledge Injection (DART & MMMED)

Questo notebook prepara le basi di conoscenza specialistiche per l'Agente MedGemma.
Oltre al dataset generalista (IMB-MCQA/QA), indicizziamo:
1.  **DART (Drug Annotation):** Per la validazione farmacologica (posologie, interazioni).
2.  **MMMED:** Setup del dataset multimodale per i test di visione.

**Output:** Una nuova collezione ChromaDB chiamata `drugs_knowledge`.
*(c) Eugenio Ruggieri 13/02/2026

In [1]:
# ----------------------------------------------------------------------------------
# CELLA 0: Alienware Hardware Monitor (Clean Import)
# ----------------------------------------------------------------------------------
import psutil
import os
import sys
import warnings

# Filtro Warning NVIDIA
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

try:
    import pynvml
except ImportError:
    pynvml = None

class AlienwareMonitor:
    @staticmethod
    def print_status(phase_name="System Status"):
        try:
            cpu_percent = psutil.cpu_percent(interval=0.1)
            ram_info = psutil.virtual_memory()
            ram_used_gb = ram_info.used / (1024 ** 3)
            
            gpu_text = "GPU: N/A"
            vram_text = "VRAM: N/A"
            
            if pynvml:
                try:
                    pynvml.nvmlInit()
                    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
                    gpu_name = pynvml.nvmlDeviceGetName(handle)
                    if isinstance(gpu_name, bytes): gpu_name = gpu_name.decode('utf-8')
                    mem_info = pynvml.nvmlDeviceGetMemoryInfo(handle)
                    temp = pynvml.nvmlDeviceGetTemperature(handle, pynvml.NVML_TEMPERATURE_GPU)
                    vram_used_mb = mem_info.used / 1024 / 1024
                    gpu_text = f"🔥 GPU: {gpu_name} | Temp: {temp}°C"
                    vram_text = f"💾 VRAM: {vram_used_mb:.0f}MB / {mem_info.total/1024/1024:.0f}MB"
                    pynvml.nvmlShutdown()
                except: pass

            print(f"\n--- 🖥️  MONITOR [{phase_name}] ---")
            print(f"{gpu_text} | {vram_text}")
            print(f"🧠 RAM: {ram_used_gb:.1f}GB | CPU: {cpu_percent}%")
            print("-" * 40)
        except: pass

AlienwareMonitor.print_status("Start Notebook 1")


--- 🖥️  MONITOR [Start Notebook 1] ---
🔥 GPU: NVIDIA GeForce RTX 5070 Laptop GPU | Temp: 54°C | 💾 VRAM: 260MB / 8151MB
🧠 RAM: 3.7GB | CPU: 0.4%
----------------------------------------


In [ ]:
# ----------------------------------------------------------------------------------
# CELLA 1 (Fixed): Download Universale e Salvataggio DART & MMMED
# ----------------------------------------------------------------------------------
from datasets import load_dataset, DatasetDict, Dataset
from huggingface_hub import login
import os

# --- 1. CONFIGURAZIONE ---
# Inserisci il tuo token HF qui sotto tra virgolette se non hai già fatto login
# HF_TOKEN = "hf_..." 
# login(token=HF_TOKEN)

# PATH DEFINITIVI
PATH_BASE = "/mnt/f/processed_datasets"
PATH_DART_SAVE = os.path.join(PATH_BASE, "DART")
PATH_MMMED_SAVE = os.path.join(PATH_BASE, "MMMED")

# REPO IDs (Uso quelli che hai indicato tu nei log)
REPO_DART = "praiselab-picuslab/DART"
REPO_MMMED = "praiselab-picuslab/MMMED"

# --- 2. FUNZIONE DI SALVATAGGIO ROBUSTA ---
def download_and_save(repo_id, save_path):
    print(f"\n📥 Scaricamento: {repo_id}...")
    try:
        # Scarica il dataset
        ds = load_dataset(repo_id)
        
        # Analisi struttura per evitare errori 'train'
        info_str = ""
        if isinstance(ds, DatasetDict):
            keys = list(ds.keys())
            info_str = f"Splits trovati: {keys}"
            # Se c'è 'train', prendiamo il conteggio da lì, altrimenti dal primo split
            first_split = keys[0]
            count = len(ds[first_split])
            info_str += f" | Righe ({first_split}): {count}"
        elif isinstance(ds, Dataset):
            # È un dataset unico senza splits
            info_str = "Dataset unico (senza splits)"
            count = len(ds)
            info_str += f" | Righe: {count}"
        else:
            info_str = f"Tipo oggetto: {type(ds)}"

        # Salvataggio
        print(f"   💾 Salvataggio in: {save_path}...")
        ds.save_to_disk(save_path)
        print(f"   ✅ Successo! {info_str}")
        
    except Exception as e:
        print(f"   ❌ Errore critico con {repo_id}: {e}")

# --- 3. ESECUZIONE ---
# Verifica cartelle
os.makedirs(PATH_DART_SAVE, exist_ok=True)
os.makedirs(PATH_MMMED_SAVE, exist_ok=True)

# Download DART
download_and_save(REPO_DART, PATH_DART_SAVE)

# Monitor Call
try:
    import psutil
    print(f"\n--- 💾 Verifica Spazio Disco ---")
    hdd = psutil.disk_usage('/mnt/f')
    print(f"Spazio Libero su F: {hdd.free / (1024**3):.2f} GB")
except:
    pass

# Download MMMED
download_and_save(REPO_MMMED, PATH_MMMED_SAVE)

# Monitor Call
try:
    import psutil
    print(f"\n--- 💾 Verifica Spazio Disco ---")
    hdd = psutil.disk_usage('/mnt/f')
    print(f"Spazio Libero su F: {hdd.free / (1024**3):.2f} GB")
except:
    pass

In [ ]:
# ----------------------------------------------------------------------------------
# CELLA 1-Bis  VERIFICA DATASET : Ispezione Dataset Scaricati (Quality Check)
# ----------------------------------------------------------------------------------
from datasets import load_from_disk
import pandas as pd
import random
import os

# Percorso definitivo dei dataset scaricati da Hugging Face
PATH_DART = "/mnt/f/processed_datasets/DART"
PATH_MMMED = "/mnt/f/processed_datasets/MMMED"

def inspect_dataset(path, name):
    print(f"\n{'='*60}")
    print(f"🧐 ISPEZIONE DATASET: {name}")
    print(f"{'='*60}")
    
    if not os.path.exists(path):
        print(f"❌ ERRORE: Percorso non trovato: {path}")
        return

    try:
        # 1. Caricamento dal disco
        ds = load_from_disk(path)
        
        # Gestione Splits (Train/Test/Validation)
        # Se è un DatasetDict, prendiamo il 'train', altrimenti il dataset intero
        if hasattr(ds, 'keys'):
            available_splits = list(ds.keys())
            print(f"📂 Splits disponibili: {available_splits}")
            target_split = 'train' if 'train' in available_splits else available_splits[0]
            data = ds[target_split]
            print(f"👉 Analisi sullo split: '{target_split}'")
        else:
            data = ds
            print("👉 Dataset piatto (senza split)")

        # 2. Statistiche di Base
        print(f"📊 Numero Totale Righe: {len(data)}")
        print(f"📝 Nomi Colonne: {data.column_names}")
        
        # 3. Campionamento Random (3 righe)
        print(f"\n🎲 Campione Random (3 righe):")
        if len(data) > 3:
            random_indices = random.sample(range(len(data)), 3)
            sample = data.select(random_indices)
        else:
            sample = data
            
        # Conversione in Pandas per visualizzazione pulita
        df_sample = pd.DataFrame(sample)
        
        # Impostiamo Pandas per mostrare tutto il testo senza tagliarlo
        pd.set_option('display.max_colwidth', None)
        
        # Display (se siamo in Jupyter usa display(), altrimenti print)
        try:
            display(df_sample)
        except NameError:
            print(df_sample.to_markdown())
            
    except Exception as e:
        print(f"⚠️ Errore durante l'ispezione: {e}")

# --- ESECUZIONE VERIFICHE ---
inspect_dataset(PATH_DART, "DART (Farmacologia)")
inspect_dataset(PATH_MMMED, "MMMED (Visione Medica)")

In [5]:
# ----------------------------------------------------------------------------------
# CELLA 2: Indicizzazione DART (Real Data) in ChromaDB
# ----------------------------------------------------------------------------------
import chromadb
from chromadb.utils import embedding_functions
from datasets import load_from_disk
from tqdm import tqdm
import os

# --- CONFIGURAZIONE ---
PATH_DART_SAVE = "/mnt/f/processed_datasets/DART"
CHROMA_PATH = "/mnt/f/chroma_db_store"

# Monitor Call
try:
    import psutil
    print(f"\n--- 💾 Verifica Spazio Disco ---")
    hdd = psutil.disk_usage('/mnt/f')
    print(f"Spazio Libero su F: {hdd.free / (1024**3):.2f} GB")
except:
    pass

print("⚙️ Connessione a ChromaDB...")
client = chromadb.PersistentClient(path=CHROMA_PATH)
emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

# Creazione della collezione specifica per la Farmacologia
coll_drugs = client.get_or_create_collection(name="drugs_knowledge", embedding_function=emb_fn)

def index_dart_final(path, collection):
    if collection.count() > 0:
        print(f"✅ Collezione {collection.name} già popolata ({collection.count()} docs).")
        return

    print(f"📥 Caricamento DART dallo split 'DART'...")
    ds = load_from_disk(path)
    data = ds['DART'] # Usiamo lo split specifico trovato nell'ispezione

    batch_size = 64 # Batch più piccolo per gestire testi lunghi
    total_rows = len(data)
    
    print(f"🚀 Indicizzazione di {total_rows} farmaci. Attendere...")

    for i in tqdm(range(0, total_rows, batch_size)):
        end = min(i + batch_size, total_rows)
        batch = data[i:end]
        
        ids, documents, metadatas = [], [], []
        
        for j in range(len(batch['Nome Medicinale'])):
            # Estrazione dei campi usando i nomi esatti scoperti
            nome = batch['Nome Medicinale'][j] or "N/A"
            principio = batch['principio_attivo'][j] or "N/A"
            indicazioni = batch['04.1 Indicazioni terapeutiche'][j] or ""
            posologia = batch['04.2 Posologia e modo di somministrazione'][j] or ""
            atc = batch['Descrizione ATC'][j] or ""

            # Creazione del "Knowledge Block" per l'Agente
            # Puliamo un po' il testo per evitare token inutili
            text_block = (
                f"FARMACO: {nome}\n"
                f"PRINCIPIO ATTIVO: {principio}\n"
                f"CLASSE ATC: {atc}\n"
                f"INDICAZIONI: {indicazioni[:1500]}\n" # Tronchiamo se troppo lungo per evitare crash
                f"POSOLOGIA: {posologia[:1500]}"
            )
            
            ids.append(f"dart_{i+j}")
            documents.append(text_block)
            metadatas.append({
                "source": "DART",
                "drug_name": str(nome),
                "atc_code": str(batch['Codice ATC'][j])
            })
        
        collection.add(ids=ids, documents=documents, metadatas=metadatas)

    print(f"✅ DART indicizzato con successo in '{collection.name}'!")

index_dart_final(PATH_DART_SAVE, coll_drugs)

# Monitor Call
try:
    import psutil
    print(f"\n--- 💾 Verifica Spazio Disco ---")
    hdd = psutil.disk_usage('/mnt/f')
    print(f"Spazio Libero su F: {hdd.free / (1024**3):.2f} GB")
except:
    pass


--- 💾 Verifica Spazio Disco ---
Spazio Libero su F: 1605.12 GB
⚙️ Connessione a ChromaDB...
📥 Caricamento DART dallo split 'DART'...
🚀 Indicizzazione di 16029 farmaci. Attendere...


100%|██████████| 251/251 [06:39<00:00,  1.59s/it]

✅ DART indicizzato con successo in 'drugs_knowledge'!

--- 💾 Verifica Spazio Disco ---
Spazio Libero su F: 1604.86 GB


In [6]:
# ----------------------------------------------------------------------------------
# CELLA 3: Indicizzazione MMMED (Casi Multimodali Italiani)
# ----------------------------------------------------------------------------------
PATH_MMMED_SAVE = "/mnt/f/processed_datasets/MMMED"

# Creazione collezione per i casi clinici visivi
coll_vision = client.get_or_create_collection(name="vision_cases", embedding_function=emb_fn)

def index_mmmed_italian(path, collection):
    if collection.count() > 0:
        print(f"✅ Collezione {collection.name} già pronta.")
        return

    print(f"📥 Caricamento MMMED dallo split 'Italian'...")
    ds = load_from_disk(path)
    # Usiamo lo split 'Italian' scoperto nell'ispezione
    data = ds['Italian'] 

    ids, documents, metadatas = [], [], []

    for i in range(len(data)):
        # Creiamo un record che lega la domanda all'URL dell'immagine
        text_content = f"CATEGORIA: {data['Category'][i]}\nDOMANDA: {data['Question'][i]}"
        
        ids.append(f"mmmed_it_{i}")
        documents.append(text_content)
        metadatas.append({
            "source": "MMMED_IT",
            "image_url": data['Image url'][i],
            "correct_answer": data['Correct Answer'][i],
            "category": data['Category'][i]
        })

    collection.add(ids=ids, documents=documents, metadatas=metadatas)
    print(f"✅ {len(ids)} casi clinici MMMED (Italiano) indicizzati.")

index_mmmed_italian(PATH_MMMED_SAVE, coll_vision)
# Monitor Call
try:
    import psutil
    print(f"\n--- 💾 Verifica Spazio Disco ---")
    hdd = psutil.disk_usage('/mnt/f')
    print(f"Spazio Libero su F: {hdd.free / (1024**3):.2f} GB")
except:
    pass
AlienwareMonitor.print_status("Notebook A: Tutte le conoscenze iniettate")

📥 Caricamento MMMED dallo split 'Italian'...
✅ 194 casi clinici MMMED (Italiano) indicizzati.

--- 💾 Verifica Spazio Disco ---
Spazio Libero su F: 1604.86 GB

--- 🖥️  MONITOR [Notebook A: Tutte le conoscenze iniettate] ---
🔥 GPU: NVIDIA GeForce RTX 5070 Laptop GPU | Temp: 39°C | 💾 VRAM: 260MB / 8151MB
🧠 RAM: 5.1GB | CPU: 0.0%
----------------------------------------


In [ ]:
# ----------------------------------------------------------------------------------
# CELLA 3-bis DI VALIDAZIONE: Check Integrale di ChromaDB
# ----------------------------------------------------------------------------------
import chromadb
import pandas as pd

CHROMA_PATH = "/mnt/f/chroma_db_store"
client = chromadb.PersistentClient(path=CHROMA_PATH)

def verify_knowledge_base():
    print(f"\n{'='*60}")
    print(f"🏥  REPORT STATISTICO KNOWLEDGE BASE (SSD F:)")
    print(f"{'='*60}")
    
    # 1. Recupero lista collezioni
    colls = client.list_collections()
    if not colls:
        print("❌ Nessuna collezione trovata nel database!")
        return

    # 2. Creazione Tabella Riepilogativa
    stats = []
    for c in colls:
        count = c.count()
        # Recuperiamo un campione per vedere i metadati
        sample = c.peek(1)
        source = "N/A"
        if sample['metadatas']:
            source = sample['metadatas'][0].get('source', 'Unknown')
        
        stats.append({
            "Nome Collezione": c.name,
            "Documenti": f"{count:,}",
            "Sorgente": source
        })

    df_stats = pd.DataFrame(stats)
    print(df_stats.to_markdown(index=False))
    
    # 3. Test di Integrità Farmacologica (DART)
    print(f"\n💊 TEST RECUPERO FARMACO (DART):")
    coll_drugs = client.get_collection("drugs_knowledge")
    # Cerchiamo un farmaco che abbiamo visto nell'ispezione (es. Esomeprazolo)
    test_query = "Esomeprazolo indicazioni e reflusso"
    res_drug = coll_drugs.query(query_texts=[test_query], n_results=1)
    
    if res_drug['documents'][0]:
        print(f"✅ DART Funzionante! Recuperato: {res_drug['metadatas'][0][0]['drug_name']}")
        print(f"📄 Estratto: {res_drug['documents'][0][0][:150]}...")
    else:
        print("⚠️ Attenzione: Il recupero DART non ha restituito risultati.")

    # 4. Test di Integrità Visiva (MMMED)
    print(f"\n📸 TEST RECUPERO CASI VISIVI (MMMED):")
    coll_vision = client.get_collection("vision_cases")
    res_vis = coll_vision.query(query_texts=["Rheumatology"], n_results=1)
    
    if res_vis['documents'][0]:
        print(f"✅ MMMED Funzionante! Categoria: {res_vis['metadatas'][0][0]['category']}")
        print(f"🔗 URL Immagine: {res_vis['metadatas'][0][0]['image_url']}")
    else:
        print("⚠️ Attenzione: Il recupero MMMED non ha restituito risultati.")

    print(f"\n{'='*60}")
    try:
        AlienwareMonitor.print_status("ChromaDB Integrity Check Complete")
    except:
        pass

# Esecuzione
verify_knowledge_base()

In [ ]:
# ----------------------------------------------------------------------------------
# CELLA 4: Audit Allineamento Conoscenza (Disco vs ChromaDB)
# ----------------------------------------------------------------------------------
import os
import chromadb
from datasets import load_from_disk

# Percorsi confermati dal tuo ls -lR
PATHS = {
    "internal_knowledge": "/mnt/f/processed_datasets/IMB-MCQA/Dataset_Arrow",
    "external_knowledge": "/mnt/f/processed_datasets/IMB_Dataset_Arrow/IMB_Dataset_Safe",
    "drugs_knowledge": "/mnt/f/processed_datasets/DART/DART",
    "vision_cases": "/mnt/f/processed_datasets/MMMED/Italian"
}

client = chromadb.PersistentClient(path="/mnt/f/chroma_db_store")

print(f"{'='*80}\n🕵️  VERIFICA INTEGRITÀ DATASET MEDGEMMA\n{'='*80}")

for coll_name, disk_path in PATHS.items():
    print(f"\n📁 Analisi: {coll_name}")
    
    # 1. Check Disco
    try:
        ds = load_from_disk(disk_path)
        disk_count = len(ds)
        cols = ds.column_names
        print(f"   💾 DISCO: {disk_count:,} record | Colonne: {cols}")
    except Exception as e:
        print(f"   ❌ ERRORE DISCO: Controlla se il path è corretto. {e}")
        disk_count = 0

    # 2. Check Chroma
    try:
        coll = client.get_collection(coll_name)
        chroma_count = coll.count()
        print(f"   🔹 CHROMA: {chroma_count:,} documenti indicizzati.")
    except:
        print(f"   ⚠️  CHROMA: Collezione non trovata.")
        chroma_count = 0

    # 3. Decisione
    if disk_count > chroma_count:
        print(f"   🚨 STATO: INCOMPLETO. Mancano {disk_count - chroma_count:,} record.")
    elif disk_count == chroma_count and disk_count > 0:
        print(f"   ✅ STATO: ALLINEATO.")
    else:
        print(f"   ❓ STATO: Verificare integrità file.")

print(f"\n{'='*80}")

In [2]:
# ----------------------------------------------------------------------------------
# CELLA 5: RE-INDICIZZAZIONE INTEGRALE (IMB-MCQA & IMB-QA)
# ----------------------------------------------------------------------------------
import os
import psutil
import warnings
import chromadb
from chromadb.utils import embedding_functions
from datasets import load_from_disk
from tqdm import tqdm

# 1. SILENZIAMO I WARNING INUTILI
warnings.filterwarnings("ignore", category=FutureWarning)

# 2. DEFINIZIONE MONITOR RISORSE
try:
    import pynvml
except ImportError:
    pynvml = None

class AlienwareMonitor:
    @staticmethod
    def print_status(phase_name="Status"):
        try:
            cpu = psutil.cpu_percent()
            ram = psutil.virtual_memory().used / (1024**3)
            gpu_info = "GPU: N/A"
            if pynvml:
                pynvml.nvmlInit()
                h = pynvml.nvmlDeviceGetHandleByIndex(0)
                temp = pynvml.nvmlDeviceGetTemperature(h, 0)
                mem = pynvml.nvmlDeviceGetMemoryInfo(h)
                gpu_info = f"🔥 GPU: {temp}°C | VRAM: {mem.used/(1024**2):.0f}MB"
                pynvml.nvmlShutdown()
            print(f"\n--- 🖥️  MONITOR [{phase_name}] | {gpu_info} | 🧠 RAM: {ram:.1f}GB | CPU: {cpu}% ---")
        except: pass

# 3. CONFIGURAZIONE PERCORSI E DATABASE
PATHS = {
    "internal_knowledge": "/mnt/f/processed_datasets/IMB-MCQA/Dataset_Arrow",
    "external_knowledge": "/mnt/f/processed_datasets/IMB_Dataset_Arrow/IMB_Dataset_Safe"
}
CHROMA_PATH = "/mnt/f/chroma_db_store"

client = chromadb.PersistentClient(path=CHROMA_PATH)
emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

# 4. FUNZIONE CORE DI RICOSTRUZIONE
def rebuild_collection(coll_name, disk_path, id_prefix):
    # Tabula Rasa
    try:
        client.delete_collection(coll_name)
        print(f"\n🗑️  Collezione '{coll_name}' eliminata.")
    except: pass

    coll = client.create_collection(name=coll_name, embedding_function=emb_fn)
    
    # Caricamento Dataset
    print(f"📥 Caricamento da: {disk_path}")
    ds = load_from_disk(disk_path)
    data = ds['train'] if hasattr(ds, 'keys') and 'train' in ds else ds
    total_rows = len(data)
    
    print(f"🚀 Indicizzazione di {total_rows:,} record in '{coll_name}'...")
    
    # Batch size ottimizzato per RTX 5070 + SSD Crucial
    batch_size = 256 
    
    for i in tqdm(range(0, total_rows, batch_size)):
        end = min(i + batch_size, total_rows)
        batch = data.select(range(i, end))
        
        docs = []
        for row in batch:
            # Logica basata sulle colonne reali trovate nell'audit
            if coll_name == "internal_knowledge":
                text = (f"DOMANDA: {row['Question']}\n"
                        f"OPZIONI: A:{row['AnswerA']}, B:{row['AnswerB']}, C:{row['AnswerC']}, D:{row['AnswerD']}\n"
                        f"CORRECT: {row['Correct Answer']}")
            else:
                # Per IMB-QA usiamo Question + Answer
                text = f"Q: {row['Question']}\nA: {row['Answer']}"
            docs.append(text)

        ids = [f"{id_prefix}_{j}" for j in range(i, end)]
        metadatas = [{"source": "IMB_FULL_FIXED"} for _ in range(len(docs))]
        
        coll.add(ids=ids, documents=docs, metadatas=metadatas)
        
        # Monitoraggio ogni 100k per l'external
        if i > 0 and i % 100000 == 0:
            AlienwareMonitor.print_status(f"Progress {coll_name}")

    print(f"✅ Finito: {coll.count():,} record in {coll_name}.")

# 5. ESECUZIONE SEQUENZIALE
AlienwareMonitor.print_status("START RE-INDEXING")

# Prima la piccola (27k)
rebuild_collection("internal_knowledge", PATHS["internal_knowledge"], "int")

# Poi la grande (783k)
rebuild_collection("external_knowledge", PATHS["external_knowledge"], "ext")

AlienwareMonitor.print_status("FINISH - FULL KB READY")


--- 🖥️  MONITOR [START RE-INDEXING] | 🔥 GPU: 55°C | VRAM: 260MB | 🧠 RAM: 4.4GB | CPU: 1.3% ---

🗑️  Collezione 'internal_knowledge' eliminata.
📥 Caricamento da: /mnt/f/processed_datasets/IMB-MCQA/Dataset_Arrow
🚀 Indicizzazione di 27,100 record in 'internal_knowledge'...


100%|██████████| 106/106 [09:11<00:00,  5.21s/it]


✅ Finito: 27,100 record in internal_knowledge.

🗑️  Collezione 'external_knowledge' eliminata.
📥 Caricamento da: /mnt/f/processed_datasets/IMB_Dataset_Arrow/IMB_Dataset_Safe
🚀 Indicizzazione di 783,192 record in 'external_knowledge'...


 23%|██▎       | 691/3060 [1:17:47<4:26:43,  6.76s/it]


KeyboardInterrupt: 

In [1]:
# ----------------------------------------------------------------------------------
# CELLA: INDICIZZAZIONE MASSIVA EXTERNAL KNOWLEDGE (GPU ACCELERATED)
# --------------------------------------------------
import os
import chromadb
from chromadb.utils import embedding_functions
from datasets import load_from_disk
from tqdm import tqdm
import torch # Per verificare CUDA

# --- 1. CONFIGURAZIONE ---
EXTERNAL_PATH = "/mnt/f/processed_datasets/IMB_Dataset_Arrow/IMB_Dataset_Safe"
CHROMA_PATH = "/mnt/f/chroma_db_store"
COLLECTION_NAME = "external_knowledge"

# --- 2. VERIFICA CUDA ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Dispositivo di calcolo rilevato: {device.upper()}")
if device == "cuda":
    print(f"🚀 GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  ATTENZIONE: CUDA non rilevato. La GPU resterà ferma!")

# --- 3. INIZIALIZZAZIONE ---
client = chromadb.PersistentClient(path=CHROMA_PATH)
emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-MiniLM-L12-v2",
    device=device # <--- Usa CUDA se disponibile
)

# Bonifica (Tabula Rasa solo per l'External)
try:
    client.delete_collection(COLLECTION_NAME)
    print(f"🗑️  Collezione '{COLLECTION_NAME}' resettata.")
except:
    pass

coll_ext = client.create_collection(name=COLLECTION_NAME, embedding_function=emb_fn)

# --- 4. CARICAMENTO E INDICIZZAZIONE ---
print(f"📥 Caricamento dataset massivo da: {EXTERNAL_PATH}")
ds_ext = load_from_disk(EXTERNAL_PATH)
# Se è un DatasetDict prendi 'train', altrimenti usa ds_ext direttamente
data = ds_ext['train'] if hasattr(ds_ext, 'keys') and 'train' in ds_ext else ds_ext

total_rows = len(data)
batch_size = 256 # Batch size ideale per RTX 5070

print(f"⚙️  Inizio indicizzazione di {total_rows:,} record...")

for i in tqdm(range(0, total_rows, batch_size)):
    end = min(i + batch_size, total_rows)
    batch = data.select(range(i, end))
    
    # Estrazione Question + Answer
    documents = [f"Q: {row['Question']}\nA: {row['Answer']}" for row in batch]
    ids = [f"ext_{j}" for j in range(i, end)]
    metadatas = [{"source": "IMB_FULL_FIXED"} for _ in range(len(documents))]
    
    # Aggiunta a ChromaDB
    coll_ext.add(ids=ids, documents=documents, metadatas=metadatas)
    
    # Monitoraggio ogni 100k
    if i > 0 and i % 100000 == 0:
        try:
            AlienwareMonitor.print_status(f"External Indexing: {i}/{total_rows}")
        except:
            pass

print(f"\n✅ COMPLETATO: {coll_ext.count():,} record indicizzati nella collezione External.")

/home/eugrug/miniforge3/envs/conda_ok_13feb/lib/python3.11/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


🖥️  Dispositivo di calcolo rilevato: CUDA
🚀 GPU: NVIDIA GeForce RTX 5070 Laptop GPU
🗑️  Collezione 'external_knowledge' resettata.
📥 Caricamento dataset massivo da: /mnt/f/processed_datasets/IMB_Dataset_Arrow/IMB_Dataset_Safe
⚙️  Inizio indicizzazione di 783,192 record...


100%|██████████| 3060/3060 [2:57:43<00:00,  3.48s/it]  



✅ COMPLETATO: 783,192 record indicizzati nella collezione External.
